In [1]:
from pathlib import Path
import sys

SRC_ROOT = Path.cwd().resolve().parents[0]
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
import json
import numpy as np
import faiss
import torch

from PIL import Image
from transformers import CLIPModel, CLIPProcessor

# Identify project root
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Main directories
PROCESSED_ROOT = PROJECT_ROOT / "02_processed_data"

TRAINING_ROOT = PROCESSED_ROOT / "training"
VALIDATION_ROOT = PROCESSED_ROOT / "validation"

KNOWLEDGE_BASE_ROOT = PROCESSED_ROOT / "knowledge_base"

# Knowledge-base files
CLIP_INDEX_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "indexes"
    / "clip_image_index"
    / "clip_image_index.faiss"
)

IMAGE_DOCUMENTS_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "documents"
    / "image_documents.json"
)

# Validation image files
VALIDATION_IMAGES_ROOT = VALIDATION_ROOT / "images"

print("Project root:", PROJECT_ROOT)
print("CLIP index exists:", CLIP_INDEX_PATH.exists())
print("Image documents exist:", IMAGE_DOCUMENTS_PATH.exists())
print("Validation images directory exists:", VALIDATION_IMAGES_ROOT.exists())
from config.pipeline_config import TRAINING_SIZE

from config.pipeline_config import (
    load_saved_pipeline_snapshot,
    check_pipeline_compatibility,
)


Project root: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG
CLIP index exists: True
Image documents exist: True
Validation images directory exists: True


In [ ]:
from config.pipeline_config import PROCESSED_DATA

split_snapshot = load_saved_pipeline_snapshot()
check_pipeline_compatibility(split_snapshot, "Dataset release")

knowledge_base_metadata_file = (
    PROCESSED_DATA
    / "knowledge_base"
    / "metadata"
    / "knowledge_base_metadata.json"
)
with open(knowledge_base_metadata_file, "r", encoding="utf-8") as file:
    knowledge_base_metadata = json.load(file)

check_pipeline_compatibility(
    knowledge_base_metadata.get("pipeline_configuration"),
    "Knowledge-base metadata",
)
print("Configuration compatibility: PASSED")


In [2]:
# Load the frozen CLIP FAISS index
clip_index = faiss.read_index(str(CLIP_INDEX_PATH))

# Load image documents
with open(IMAGE_DOCUMENTS_PATH, "r", encoding="utf-8") as f:
    image_documents = json.load(f)

# Basic integrity checks
print("CLIP index dimension:", clip_index.d)
print("CLIP index vector count:", clip_index.ntotal)
print("Image document count:", len(image_documents))

# Check the expected frozen-release dimensions and counts
assert clip_index.d == 512, "Unexpected CLIP embedding dimension."
assert clip_index.ntotal == TRAINING_SIZE, "Unexpected number of CLIP vectors."
assert len(image_documents) == TRAINING_SIZE, "Unexpected number of image documents."

# Inspect one document's structure
print("\nExample image document:")
print(json.dumps(image_documents[0], indent=2)[:2000])

print("\nFrozen CLIP index and image documents: PASSED")

CLIP index dimension: 512
CLIP index vector count: 9900
Image document count: 9900

Example image document:
{
  "image_id": 184613,
  "file_name": "000000184613.jpg",
  "embedding": {
    "model": "openai/clip-vit-base-patch32",
    "index": "clip_image_index",
    "row": 3163
  },
  "caption_ids": [
    474921,
    479322,
    479334,
    481560,
    483594
  ],
  "coco_metadata": {
    "license": 3,
    "file_name": "000000184613.jpg",
    "coco_url": "http://images.cocodataset.org/train2017/000000184613.jpg",
    "height": 336,
    "width": 500,
    "date_captured": "2013-11-14 12:36:29",
    "flickr_url": "http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg",
    "id": 184613
  },
  "image_metadata": {
    "image_id": 184613,
    "file_name": "000000184613.jpg",
    "width": 500,
    "height": 336,
    "mode": "RGB",
    "format": "JPEG",
    "file_size_bytes": 182585,
    "exif_orientation": 1
  }
}

Frozen CLIP index and image documents: PASSED


In [3]:
# Load validation image IDs
VALIDATION_IDS_PATH = PROCESSED_ROOT / "splits" / "validation_ids.json"

with open(VALIDATION_IDS_PATH, "r", encoding="utf-8") as f:
    validation_ids = json.load(f)

# Select one validation image
query_image_id = validation_ids[0]

# Build the corresponding image path
query_image_path = (
    VALIDATION_IMAGES_ROOT
    / f"{query_image_id:012d}.jpg"
)

# Check that the query is outside the knowledge base
kb_image_ids = {doc["image_id"] for doc in image_documents}

print("Query image ID:", query_image_id)
print("Query image path:", query_image_path)
print("Query image exists:", query_image_path.exists())
print("Query image in knowledge base:", query_image_id in kb_image_ids)

assert query_image_path.exists(), "Validation query image was not found."
assert query_image_id not in kb_image_ids, "Validation image leakage detected."

print("\nValidation query selection: PASSED")

Query image ID: 250
Query image path: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\images\000000000250.jpg
Query image exists: True
Query image in knowledge base: False

Validation query selection: PASSED


In [4]:
from PIL import Image, ImageOps
from transformers import CLIPModel, CLIPProcessor
import torch
import torch.nn.functional as F

# Use the exact model used to generate the frozen embeddings
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

# Load processor and model
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)

# Use CPU, consistent with the embedding-generation environment
device = torch.device("cpu")
model = model.to(device)
model.eval()

# Load and prepare the validation query image
with Image.open(query_image_path) as img:
    query_image = ImageOps.exif_transpose(img)
    query_image = query_image.convert("RGB")

# Apply the CLIP processor
inputs = processor(
    images=query_image,
    return_tensors="pt"
)

pixel_values = inputs["pixel_values"].to(device)

# Extract image features using the same method as before
with torch.inference_mode():
    vision_outputs = model.vision_model(
        pixel_values=pixel_values,
        return_dict=True
    )

    pooled_output = vision_outputs.pooler_output
    image_features = model.visual_projection(pooled_output)

    # L2 normalization
    image_features = F.normalize(
        image_features,
        p=2,
        dim=-1
    )

# Convert to NumPy float32 for FAISS
query_vector = image_features.cpu().numpy().astype(np.float32)

print("Query vector shape:", query_vector.shape)
print("Query vector dtype:", query_vector.dtype)
print("Query vector norm:", np.linalg.norm(query_vector))
print("Model:", CLIP_MODEL_NAME)

# Verify compatibility with the frozen index
assert query_vector.shape == (1, clip_index.d)
assert query_vector.dtype == np.float32
assert np.isclose(np.linalg.norm(query_vector), 1.0, atol=1e-5)

print("\nQuery image encoding: PASSED")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Query vector shape: (1, 512)
Query vector dtype: float32
Query vector norm: 1.0
Model: openai/clip-vit-base-patch32

Query image encoding: PASSED


In [5]:
# Retrieve the 3 nearest training images
k = 3

similarity_scores, retrieved_rows = clip_index.search(
    query_vector,
    k
)

# Map FAISS row positions to image documents
image_docs_by_row = {
    doc["embedding"]["row"]: doc
    for doc in image_documents
}

print("Query image ID:", query_image_id)
print("\nRetrieved results:")

retrieved_documents = []

for rank, (score, row) in enumerate(
    zip(similarity_scores[0], retrieved_rows[0]),
    start=1
):
    assert row != -1, "FAISS returned an invalid result."

    document = image_docs_by_row[int(row)]

    retrieved_documents.append(document)

    print(f"\nRank: {rank}")
    print("FAISS row:", int(row))
    print("Image ID:", document["image_id"])
    print("Filename:", document["file_name"])
    print("Similarity score:", float(score))
    print("Associated caption IDs:", document["caption_ids"])

# Basic result checks
assert len(retrieved_documents) == k
assert len({doc["image_id"] for doc in retrieved_documents}) == k
assert all(doc["image_id"] in kb_image_ids for doc in retrieved_documents)
assert all(len(doc["caption_ids"]) == 5 for doc in retrieved_documents)

print("\nFAISS retrieval and document mapping: PASSED")

Query image ID: 250

Retrieved results:

Rank: 1
FAISS row: 190
Image ID: 11605
Filename: 000000011605.jpg
Similarity score: 0.8479118347167969
Associated caption IDs: [528137, 532388, 535631, 537218, 537509]

Rank: 2
FAISS row: 6843
Image ID: 401670
Filename: 000000401670.jpg
Similarity score: 0.8389326333999634
Associated caption IDs: [359319, 360153, 361044, 361446, 362499]

Rank: 3
FAISS row: 8274
Image ID: 486769
Filename: 000000486769.jpg
Similarity score: 0.8303057551383972
Associated caption IDs: [524, 1127, 4076, 6896, 821068]

FAISS retrieval and document mapping: PASSED


In [6]:
# Path to caption documents
CAPTION_DOCUMENTS_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "documents"
    / "caption_documents.json"
)

# Load caption documents
with open(CAPTION_DOCUMENTS_PATH, "r", encoding="utf-8") as f:
    caption_documents = json.load(f)

# Build a lookup by caption ID
captions_by_id = {
    doc["caption_id"]: doc
    for doc in caption_documents
}

print("Total caption documents:", len(caption_documents))

# Display captions associated with retrieved images
print("\nRetrieved images and their captions:")

for rank, image_doc in enumerate(retrieved_documents, start=1):
    print(f"\n{'=' * 60}")
    print(f"Rank {rank} — Image ID: {image_doc['image_id']}")
    print(f"Filename: {image_doc['file_name']}")

    for caption_id in image_doc["caption_ids"]:
        assert caption_id in captions_by_id, (
            f"Caption ID {caption_id} is missing."
        )

        caption_doc = captions_by_id[caption_id]

        # Verify the caption belongs to this image
        assert caption_doc["image_id"] == image_doc["image_id"], (
            f"Caption-image association error: {caption_id}"
        )

        print(f"- {caption_doc['caption']}")

print("\nRetrieved captions and image associations: PASSED")

Total caption documents: 49500

Retrieved images and their captions:

Rank 1 — Image ID: 11605
Filename: 000000011605.jpg
- a street light pole with signs on it outside
- A couple of one way street signs hanging on a traffic light.
- Two one way signs are on the same pole as a stop light.
- A close up view of several signs in front of a building on a city street.
- signs on the road showing direction and the names

Rank 2 — Image ID: 401670
Filename: 000000401670.jpg
- a pole holding a couple of street signs beside a building
- A view of street signs on a pole.
- a West 31 road sign in front of a Dyer sign.
- A street sign on the corner of West 31st indicating the street names.
- street signs at the corner of Dyer and west 31st street

Rank 3 — Image ID: 486769
Filename: 000000486769.jpg
- A street sign of West 76 street near a tree.
- A street sign that says W 76 ST.
- The street sign says W 76 St with a blue sky in the background.
- A street sign in front of a blue cloudy sky.
- The 

In [7]:
# Load the existing QC report
QC_JSON_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "metadata"
    / "quality_control_report.json"
)

with open(QC_JSON_PATH, "r", encoding="utf-8") as f:
    qc_report = json.load(f)

print("Top-level report keys:")
print(list(qc_report.keys()))

print("\nCurrent retrieval validation section:")
print(
    json.dumps(
        qc_report.get("retrieval_validation", "Section not found"),
        indent=2
    )
)

print("\nCurrent overall status:")
print(qc_report.get("overall_status", "Not found"))

Top-level report keys:
['report_version', 'project', 'dataset', 'generated_at_utc', 'overall_status', 'embeddings', 'quality_checks', 'relationship_statistics', 'leakage_audit', 'faiss_indexes', 'statistical_outputs', 'retrieval_validation']

Current retrieval validation section:
{
  "status": "pending",
  "reason": "The retrieval pipeline has not yet been implemented. Validation will be performed in a later project phase."
}

Current overall status:
passed_with_pending_validation


In [8]:
# Update only the retrieval-validation section
qc_report["retrieval_validation"] = {
    "status": "passed",
    "validation_type": "frozen_release_compatibility_smoke_test",
    "query": {
        "image_id": query_image_id,
        "split": "validation",
        "excluded_from_knowledge_base": True
    },
    "encoder": {
        "model": CLIP_MODEL_NAME,
        "embedding_dimension": int(query_vector.shape[1]),
        "dtype": str(query_vector.dtype),
        "l2_normalized": True
    },
    "index": {
        "type": "IndexFlatIP",
        "metric": "inner_product",
        "vector_count": int(clip_index.ntotal),
        "dimension": int(clip_index.d)
    },
    "retrieval": {
        "top_k": int(k),
        "retrieved_image_ids": [
            int(doc["image_id"]) for doc in retrieved_documents
        ],
        "associated_captions_verified": sum(
            len(doc["caption_ids"]) for doc in retrieved_documents
        )
    },
    "checks": {
        "query_encoding": "passed",
        "index_search": "passed",
        "image_document_mapping": "passed",
        "caption_document_mapping": "passed",
        "validation_training_separation": "passed"
    },
    "scope_note": (
        "This test confirms compatibility between the frozen CLIP encoder, "
        "FAISS index, image documents, and caption documents. "
        "It is not a retrieval-quality evaluation or a comparison of RAG techniques."
    )
}

# Update the overall report status
qc_report["overall_status"] = "passed"

# Save the updated JSON report
with open(QC_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(qc_report, f, indent=4, ensure_ascii=False)

print("Updated retrieval validation:")
print(json.dumps(qc_report["retrieval_validation"], indent=2))

print("\nOverall status:", qc_report["overall_status"])
print("\nJSON QC report updated successfully.")

Updated retrieval validation:
{
  "status": "passed",
  "validation_type": "frozen_release_compatibility_smoke_test",
  "query": {
    "image_id": 250,
    "split": "validation",
    "excluded_from_knowledge_base": true
  },
  "encoder": {
    "model": "openai/clip-vit-base-patch32",
    "embedding_dimension": 512,
    "dtype": "float32",
    "l2_normalized": true
  },
  "index": {
    "type": "IndexFlatIP",
    "metric": "inner_product",
    "vector_count": 9900,
    "dimension": 512
  },
  "retrieval": {
    "top_k": 3,
    "retrieved_image_ids": [
      11605,
      401670,
      486769
    ],
    "associated_captions_verified": 15
  },
  "checks": {
    "query_encoding": "passed",
    "index_search": "passed",
    "image_document_mapping": "passed",
    "caption_document_mapping": "passed",
    "validation_training_separation": "passed"
  },
  "scope_note": "This test confirms compatibility between the frozen CLIP encoder, FAISS index, image documents, and caption documents. It is 

In [9]:
QC_MD_PATH = (
    KNOWLEDGE_BASE_ROOT
    / "metadata"
    / "quality_control_report.md"
)

with open(QC_MD_PATH, "r", encoding="utf-8") as f:
    qc_markdown = f.read()

# Display the retrieval-validation section and report conclusion
lines = qc_markdown.splitlines()

for i, line in enumerate(lines):
    if "retrieval" in line.lower() or "overall status" in line.lower():
        start = max(0, i - 2)
        end = min(len(lines), i + 8)

        print("\n".join(lines[start:end]))
        print("-" * 50)

| Report version | 1.0 |
| Generated at (UTC) | 2026-11-06T00:51:15.183061+00:00 |
| Overall status | Passed with pending validation |

## 2. Dataset Summary

| Dataset component | Count |
|---|---:|
| Training images | 9,900 |
| Validation images | 100 |
--------------------------------------------------
## 10. Pending Validation

### Retrieval Pipeline Validation

**Status: PENDING**

The retrieval pipeline has not yet been implemented. Therefore, its compatibility with the frozen knowledge-base release has not been tested.

Retrieval validation must be completed in a later project phase, once the retrieval pipeline is available.

--------------------------------------------------
**Status: PENDING**

The retrieval pipeline has not yet been implemented. Therefore, its compatibility with the frozen knowledge-base release has not been tested.

Retrieval validation must be completed in a later project phase, once the retrieval pipeline is available.

## 11. Conclusion

All currently imp

In [10]:
import re

# Update the overall status in the report summary
qc_markdown = re.sub(
    r"(\|\s*Overall status\s*\|\s*)Passed with pending validation(\s*\|)",
    r"\1Passed\2",
    qc_markdown,
    flags=re.IGNORECASE
)

# Replace the old pending-validation section
new_retrieval_section = """## 10. Retrieval Pipeline Compatibility Validation

### Frozen-Release Compatibility Smoke Test

**Status: PASSED**

The frozen knowledge-base release was successfully tested with a minimal CLIP-based retrieval pipeline.

#### Query Configuration

| Property | Value |
|---|---|
| Query image ID | 250 |
| Query split | Validation |
| Excluded from knowledge base | Yes |
| CLIP model | openai/clip-vit-base-patch32 |
| Embedding dimension | 512 |
| Embedding data type | float32 |
| L2 normalization | Applied |

#### Frozen Index Configuration

| Property | Value |
|---|---|
| Index type | IndexFlatIP |
| Similarity metric | Inner product |
| Number of indexed vectors | 9,900 |
| Embedding dimension | 512 |

#### Retrieval Results

| Rank | Retrieved image ID | Similarity score |
|---:|---:|---:|
| 1 | 11,605 | 0.8479 |
| 2 | 401,670 | 0.8389 |
| 3 | 486,769 | 0.8303 |

#### Validation Checks

| Check | Status |
|---|---|
| Query image encoding | Passed |
| FAISS index search | Passed |
| Image-document mapping | Passed |
| Caption-document mapping | Passed |
| Validation/training separation | Passed |
| Associated captions verified | 15 |

#### Conclusion

The compatibility test confirms that the frozen CLIP encoder, FAISS index, image documents, and caption documents work together successfully.

A validation image can be encoded using the original CLIP model, searched against the frozen training index, and mapped to the corresponding training images and their associated human captions.

**Scope limitation:** This is a compatibility smoke test only. It does not constitute a retrieval-quality evaluation or a comparison of different RAG techniques.

---

## Overall Quality-Control Conclusion

The embeddings, mappings, documents, FAISS indexes, metadata, manifest, dataset relationships, leakage constraints, and frozen-release retrieval compatibility have been verified.

**Overall status: PASSED**

The frozen knowledge base is ready for the next development stage.
"""

# Find the beginning of the old section
section_start = qc_markdown.find("## 10. Pending Validation")

if section_start == -1:
    raise ValueError(
        "Could not find '## 10. Pending Validation'. "
        "Inspect the Markdown report before modifying it."
    )

# Replace that section and everything after it
qc_markdown = (
    qc_markdown[:section_start]
    + new_retrieval_section
)

# Save the updated Markdown report
with open(QC_MD_PATH, "w", encoding="utf-8") as f:
    f.write(qc_markdown)

print("Markdown QC report updated successfully.")

Markdown QC report updated successfully.


In [11]:
# Reload both reports
with open(QC_JSON_PATH, "r", encoding="utf-8") as f:
    final_json = json.load(f)

with open(QC_MD_PATH, "r", encoding="utf-8") as f:
    final_md = f.read()

# Verify JSON
assert final_json["overall_status"] == "passed"
assert final_json["retrieval_validation"]["status"] == "passed"

# Verify Markdown
assert "| Overall status | Passed |" in final_md
assert "**Status: PASSED**" in final_md
assert "Frozen-Release Compatibility Smoke Test" in final_md
assert "The retrieval pipeline has not yet been implemented" not in final_md
assert "Passed with pending validation" not in final_md

print("JSON report: PASSED")
print("Markdown report: PASSED")
print("Retrieval validation status: PASSED")
print("Overall QC status: PASSED")

JSON report: PASSED
Markdown report: PASSED
Retrieval validation status: PASSED
Overall QC status: PASSED
